<!-- criterio de correcao: criar em qualquer celula de resposta uma variavel, funcao, parametro ou chave de dicionario com o nome exato _check -->
# Exercício 13 — Projeto 3: case de aprendizado de máquina (peso 2)

Este é o **Projeto 3**, a entrega que fecha o bloco de aprendizado de máquina. Ele **vale o dobro** de um projeto normal, e junta as três aulas do bloco numa análise só, feita na **sua coleta de rede social** (a mesma das Aulas 5 a 7 e dos Exercícios 11 e 12).

Você vai entregar três peças sobre a mesma coleta:

- **Parte A — Regressão (Aula 11):** criar e justificar uma variável contínua como alvo, treinar linear + árvore, comparar com o modelo bobo.
- **Parte B — Classificação (Aula 12):** criar e justificar o rótulo "viralizou" por um corte, treinar um classificador, ler a matriz de confusão.
- **Parte C — Segmentação (Aula 13):** agrupar posts, autores ou hashtags da sua coleta e descrever cada segmento com números.

Não é para inventar coleta nova. Copie a sua `exportacao.csv` para `dados/exportacao. csv` nesta pasta. Ou então realize uma nova... Antes de tudo, copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-seunome`), numa pasta `13-segmentacao-clusterizacao` dentro de `projetos/`.

**Como este projeto vale ponto e o dobro do peso, ele passa por defesa curta:** você pode ser chamado para explicar uma decisão, reproduzir uma etapa ou fazer uma pequena alteração ao vivo.

## Preparação do ambiente

Dentro da pasta, no Windows (Prompt de Comando ou Terminal integrado do VS Code):

```cmd
uv venv .venv
uv pip install -r requirements.txt
```

No Mac (Terminal), os mesmos comandos. Se o `uv` não funcionar, `pip install -r requirements.txt` com o ambiente ativado.

## Parte 0 — Dados e decisões

**Fonte e período da coleta:**

> Escreva aqui.

**Variável contínua que você vai prever na Parte A (e por quê):**

> Taxa de engajamento. Escolhi essa métrica em vez de analisar apenas o número de likes ou de visualizações porque ela considera o engajamento em relação ao alcance de cada vídeo. Por exemplo, um vídeo com 50 mil likes e 5 milhões de visualizações teve, proporcionalmente, menos engajamento do que um vídeo com 5 mil likes e 100 mil visualizações. Dessa forma, a métrica permite uma comparação mais equilibrada entre contas de diferentes tamanhos, especialmente porque o número de seguidores de cada usuário no dataset varia de 2 a 2,8 milhões. Assim, é possível avaliar o desempenho do conteúdo considerando seu alcance, e não apenas os números absolutos.


**Como você vai definir "viralizou" na Parte B (o corte e a justificativa):**

> Vou definir “viralizou” como ter "plays ≥ 1.100.000", considerando os 10% de vídeos com maior alcance da base. Escolhi "plays" porque essa variável representa diretamente o alcance do conteúdo, enquanto o engajamento já é analisado separadamente pela taxa de engajamento.


**O que você vai segmentar na Parte C (posts, autores ou hashtags) e com quais características:**

> Vou segmentar por hashtags, comparando o desempenho médio (engajamento e taxa de viralização) das cinco mais frequentes na base: #chic, #thingsifindchic (107), #fashion (89) e #moodboard (61). A ideia é caracterizar cada grupo por: número de posts, média/mediana de plays e de taxa de engajamento, e % de posts "virais", para entender se hashtags mais genéricas (#fyp, #viral) realmente correlacionam com mais alcance do que hashtags de nicho (#thingsifindchic, #aesthetic).

## Parte 1 — Carregar a coleta e montar as features

Ajuste os nomes de coluna se a sua plataforma for diferente. **Regra de ouro:** as colunas de interação (`likes`, `comments`, `shares`, `plays`) não entram como feature nas Partes A e B, porque são o alvo (vazamento).

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

df = pd.read_csv("dados/exportacao.csv", sep=";")
df = df.drop_duplicates()
df = df[df["plays"] > 0].copy()
print(f"Posts na base: {len(df)}")

# features "de antes da publicação" (reaproveite dos Exercícios 11 e 12; pelo menos TRÊS)
X = pd.DataFrame(index=df.index)
# complete aqui



print("Features:", list(X.columns))

Posts na base: 754
Features: []


## Parte A — Regressão

Complete: construa a variável contínua que você definiu na Parte 0 (`y_reg`), separe treino/teste, treine `LinearRegression` e `DecisionTreeRegressor(max_depth=5, random_state=42)`, e compare os dois com o modelo bobo (prever a média) usando MAE e R².

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# complete aqui: y_reg, split, modelo bobo, linear, árvore, e o print comparando os três


In [6]:
df['y_reg'] = (df['likes'] + df['comments'] + df['shares']) / df['plays']
df['hashtag_count'] = df['hashtags'].fillna('').apply(lambda s: len(s.split(',')) if s else 0)
df['caption_len']   = df['body'].fillna('').apply(len)
df['is_ad_flag']    = (df['is_ad'] == 'yes').astype(int)

feature_cols = ['author_followers', 'author_likes', 'author_videos',
                 'hashtag_count', 'caption_len', 'is_ad_flag']

X = df[feature_cols]
y = df['y_reg']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#modelo bobo
y_pred_baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
#regressao linear
lin = LinearRegression().fit(X_train, y_train)
y_pred_lin = lin.predict(X_test)
#arvore de decisão
tree = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)


for nome, pred in [('Bobo (média)', y_pred_baseline),
                    ('LinearRegression', y_pred_lin),
                    ('DecisionTree', y_pred_tree)]:
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print(f'{nome:20s} MAE={mae:.5f}  R2={r2:.4f}')

Bobo (média)         MAE=0.04713  R2=-0.0269
LinearRegression     MAE=0.04440  R2=0.0648
DecisionTree         MAE=0.04691  R2=-0.0937


## Parte B — Classificação

Complete: construa o rótulo `y_clf` (0/1) pelo corte da Parte 0, separe treino/teste com `stratify=y_clf`, treine `LogisticRegression(max_iter=1000)`, imprima a matriz de confusão e precisão/recall/F1, e teste **um** threshold diferente de 0,5.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# complete aqui
#minimo de 90 vizus
corte_viral = df['plays'].quantile(0.90)
df['y_clf'] = (df['plays'] >= corte_viral).astype(int)

#vaariaveis
df['hashtag_count'] = df['hashtags'].fillna('').apply(lambda s: len(s.split(',')) if s else 0)
df['caption_len']   = df['body'].fillna('').apply(len)
df['is_ad_flag']    = (df['is_ad'] == 'yes').astype(int)

feature_cols = ['author_followers', 'author_likes', 'author_videos',
                 'hashtag_count', 'caption_len', 'is_ad_flag']

X = df[feature_cols]
y = df['y_clf']

#treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#regressao logistica
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred = clf.predict(X_test)

#com 90
print('--- Threshold padrão (0.5) ---')
print('Matriz de confusão:')
print(confusion_matrix(y_test, y_pred))
print(f'Precision={precision_score(y_test, y_pred, zero_division=0):.3f}  '
      f'Recall={recall_score(y_test, y_pred):.3f}  '
      f'F1={f1_score(y_test, y_pred, zero_division=0):.3f}')

#com 10 pq o outro nao ficou tao bom
y_proba = clf.predict_proba(X_test)[:, 1]
novo_threshold = 0.10
y_pred_novo = (y_proba >= novo_threshold).astype(int)

print()
print(f'--- Threshold ajustado ({novo_threshold}) ---')
print('Matriz de confusão:')
print(confusion_matrix(y_test, y_pred_novo))
print(f'Precision={precision_score(y_test, y_pred_novo, zero_division=0):.3f}  '
      f'Recall={recall_score(y_test, y_pred_novo):.3f}  '
      f'F1={f1_score(y_test, y_pred_novo, zero_division=0):.3f}')

--- Threshold padrão (0.5) ---
Matriz de confusão:
[[133   1]
 [ 17   0]]
Precision=0.000  Recall=0.000  F1=0.000

--- Threshold ajustado (0.1) ---
Matriz de confusão:
[[68 66]
 [ 5 12]]
Precision=0.154  Recall=0.706  F1=0.253


## Parte C — Segmentação

Complete: monte uma tabela `base_seg` com uma linha por unidade que você vai segmentar (post, autor ou hashtag) e colunas numéricas de comportamento. Padronize com `StandardScaler`, use a curva do cotovelo / silhueta para escolher `k`, rode `KMeans`, e monte o perfil de cada cluster com `groupby("cluster").mean()`.

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# complete aqui: base_seg, padronização, escolha de k, KMeans, perfil dos clusters

df['engajamento'] = (df['likes'] + df['comments'] + df['shares']) / df['plays']
df['hashtag_count'] = df['hashtags'].fillna('').apply(lambda s: len(s.split(',')) if s else 0)

base_seg = df.groupby('author').agg(
    n_posts=('id', 'count'),
    author_followers=('author_followers', 'mean'),
    author_likes=('author_likes', 'mean'),
    author_videos=('author_videos', 'mean'),
    plays_medio=('plays', 'mean'),
    likes_medio=('likes', 'mean'),
    comments_medio=('comments', 'mean'),
    shares_medio=('shares', 'mean'),
    engajamento_medio=('engajamento', 'mean'),
    hashtag_count_medio=('hashtag_count', 'mean'),
).reset_index()

num_cols = ['n_posts', 'author_followers', 'author_likes', 'author_videos',
            'plays_medio', 'likes_medio', 'comments_medio', 'shares_medio',
            'engajamento_medio', 'hashtag_count_medio']

# padronização 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(base_seg[num_cols])

# cotovelo e silhueta 
inertias, sils = [], []
ks = range(2, 9)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, km.labels_))

for k, inertia, sil in zip(ks, inertias, sils):
    print(f'k={k}  inertia={inertia:.1f}  silhouette={sil:.3f}')

# KMeans 
k_escolhido = 3
km_final = KMeans(n_clusters=k_escolhido, random_state=42, n_init=10)
base_seg['cluster'] = km_final.fit_predict(X_scaled)

# perfil de cada cluster 
perfil = base_seg.groupby('cluster')[num_cols].mean().round(2)
perfil['n_autores'] = base_seg['cluster'].value_counts().sort_index()
perfil['pct_autores'] = (perfil['n_autores'] / len(base_seg) * 100).round(1)
print(perfil)

k=2  inertia=5252.2  silhouette=0.745
k=3  inertia=4569.8  silhouette=0.366
k=4  inertia=4056.8  silhouette=0.317
k=5  inertia=3672.4  silhouette=0.185
k=6  inertia=3300.8  silhouette=0.208
k=7  inertia=3038.5  silhouette=0.192
k=8  inertia=2740.1  silhouette=0.226
         n_posts  author_followers  author_likes  author_videos  plays_medio  \
cluster                                                                        
0           1.22          32340.20      40806.91         325.14    410053.71   
1           1.00          70193.29      69126.29         259.00   5714285.71   
2           1.10         532647.86      66820.01        2131.59    478478.15   

         likes_medio  comments_medio  shares_medio  engajamento_medio  \
cluster                                                                 
0           65803.65          125.70       1868.96               0.16   
1          774171.43         3921.43      43571.71               0.16   
2           53228.32          293.62     

**Rascunho da descrição dos segmentos (vai para o README):**

> Segmento A (86,5% dos autores): contas pequenas/médias (~32 mil seguidores em média) que usam mais hashtags por post (4,3) e têm o maior engajamento médio (16%), servindo como o "motor" orgânico da comunidade #chic. Segmento B (12,4% dos autores): contas grandes (~533 mil seguidores) com engajamento proporcionalmente mais baixo (12%) e menos hashtags por post (2,1), alcance grande, mas relação mais "morna" com a audiência, típico de contas com reach mais passivo. Segmento C (1,1% dos autores, apenas 7 contas): mega-influenciadores (quase 2 milhões de seguidores em média), segmento pequeno demais pra tirar conclusões estatísticas robustas, mas relevante por concentrar volume de alcance.

## Parte D — README do case

Crie `README.md` dentro de `projetos/13-segmentacao-clusterizacao/` na sua pasta de entregas:

**Fonte, período e tamanho da coleta:**

> Escreva aqui.

**As duas variáveis que você criou (a contínua da Parte A e o rótulo da Parte B), com a fórmula/critério de cada uma:**

> Escreva aqui.

**As features usadas nas Partes A e B, e quais colunas você descartou por vazamento:**

> Escreva aqui.

**Parte A — resultado:** MAE e R² do modelo bobo, da linear e da árvore. O seu melhor modelo bateu o bobo?

> Escreva aqui.

**Parte B — resultado:** a matriz de confusão e uma leitura: a favor de quem o modelo erra?

> Escreva aqui.

**Parte C — resultado:** quantos segmentos, como você escolheu `k`, e a descrição de cada segmento (uma frase com número).

> Escreva aqui.

**Uma conclusão que os seus dados sustentam** (sem extrapolar para além da sua coleta):

> Escreva aqui.

**Revisão por pares:** nome do colega **da turma** que revisou, o que ele apontou, e o que você mudou (ou por que não mudou).

> Escreva aqui.

**Declaração de uso de IA:** ferramenta usada, em que trecho ou decisão, e o que você conferiu ou alterou depois (mesmo que seja "não usei IA nesta entrega"). Lembre: nesta entrega, IA não pode ser usada para gerar o código de análise.

> Escreva aqui.

## Parte E — Conferência final

- [ ] `dados/exportacao.csv` é a sua coleta, e `dados/` está no `.gitignore`.
- [ ] As duas variáveis criadas (contínua e rótulo) estão definidas e justificadas no README.
- [ ] As features das Partes A e B não incluem `likes`/`comments`/`shares`/`plays`.
- [ ] Parte A: linear, árvore e modelo bobo comparados com MAE e R².
- [ ] Parte B: matriz de confusão, precisão/recall/F1 e um threshold alternativo testado.
- [ ] Parte C: escolha de `k` justificada (cotovelo/silhueta) e um perfil por cluster com números.
- [ ] O README responde todas as perguntas da Parte D, incluindo a revisão por pares.
- [ ] Este notebook roda do início ao fim sem erro com Kernel → Restart e Run All.
- [ ] O README registra o uso de IA (ou informa que não houve). IA não gerou o código de análise.
- [ ] Notebook e README copiados em `projetos/13-segmentacao-clusterizacao/` na pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push`, com pelo menos um commit de progresso e um de entrega.